# UrbanHub — Météo Open-Meteo (Rémi)
Open-Meteo Historical Weather API, modèle ECMWF IFS : Paris, Lyon et Lille. Données modélisées, horaires UTC. Les dates récentes sont actualisées en relançant le script. Ce notebook décrit le flux ; le [bilan de référence 2022–2026](../docs/RESULTATS_NOAA.md) reprend les résultats du notebook 07 sur les archives enrichies en visibilité. Voir [le README](../README.md) pour les chemins de publication.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/openmeteo_batch.py').is_file())
files = sorted((root / 'data').glob('openmeteo-2020-01-01_*/gold/kpis/weather_hourly.parquet'))
if not files:
    raise FileNotFoundError('Lancer python scripts/openmeteo_batch.py depuis la racine')
hourly = pd.read_parquet(files[-1])
daily = pd.read_parquet(files[-1].with_name('weather_daily.parquet'))
display(hourly.groupby('city_id').agg(debut=('hour_utc', 'min'), fin=('hour_utc', 'max'), heures=('hour_utc', 'size'), visibilite=('visibility', 'count')))

## Saisonnalité, extrêmes et épisodes inhabituels
Calcul sur les années civiles complètes. Les seuils P95 des maxima et P5 des minima quotidiens sont calculés par ville et saison. Les jours franchissant le même seuil à la suite forment un épisode. Il s’agit d’extrêmes relatifs à la période et au modèle, pas de normales climatiques ni de records observés en station.

In [ ]:
year = hourly['hour_utc'].dt.year
days = daily.groupby([daily['day_utc'].dt.year, 'city_id']).size().unstack()
complete_years = days.index[days.min(axis=1).eq([366 if pd.Timestamp(y, 1, 1).is_leap_year else 365 for y in days.index]) & days.index.to_series().lt(year.max())]
series = hourly.loc[year.isin(complete_years)].copy()
series['month'] = series['hour_utc'].dt.month
monthly = series.pivot_table(index='month', columns='city_id', values='temperature')
monthly.plot(title='Température moyenne par mois (années complètes)', ylabel='°C', xticks=range(1, 13))
plt.show()
days = daily.loc[daily['day_utc'].dt.year.isin(complete_years) & daily['temperature_hours'].eq(24)].copy()
days['season'] = (days['day_utc'].dt.month % 12) // 3
group = days.groupby(['city_id', 'season'])
days['p95'] = group['temperature_max'].transform('quantile', q=.95)
days['p05'] = group['temperature_min'].transform('quantile', q=.05)
display(days.loc[days.groupby('city_id')['temperature_max'].idxmax(), ['city_id', 'day_utc', 'temperature_max']])
display(days.loc[days.groupby('city_id')['temperature_min'].idxmin(), ['city_id', 'day_utc', 'temperature_min']])
events = pd.concat([days[days['temperature_max'] > days['p95']].assign(type='Chaud'), days[days['temperature_min'] < days['p05']].assign(type='Froid')]).sort_values(['city_id', 'type', 'day_utc'])
events['episode'] = events.groupby(['city_id', 'type'])['day_utc'].diff().ne(pd.Timedelta(days=1)).cumsum()
display(events.groupby(['city_id', 'type', 'episode']).agg(debut=('day_utc', 'min'), fin=('day_utc', 'max'), jours=('day_utc', 'size')).sort_values('jours', ascending=False).head(10).reset_index())

## Visibilité
La visibilité est indisponible dans l’archive IFS de ce lot en 2020–2024 et partielle en 2025. Les corrélations portent uniquement sur les heures pour lesquelles elle est fournie ; aucune valeur n’est imputée.

In [ ]:
rows = []
for city, frame in hourly.groupby('city_id'):
    for variable in ['temperature', 'wind_speed', 'precipitation']:
        pairs = frame[[variable, 'visibility']].dropna()
        r = pairs[variable].corr(pairs['visibility']) if len(pairs) > 1 and pairs.nunique().min() > 1 else float('nan')
        rows.append({'city_id': city, 'variable': variable, 'heures': len(pairs), 'pearson_r': r})
display(pd.DataFrame(rows).round(3))